In [3]:
# ==============================================================================
# Test‑Time Adaptation for Low‑Resource Handwritten Character Recognition
# AIM: Improve robustness under distribution shift without retraining
# OBJECTIVES: Compare YorubaNet vs AlexNet, with/without TTA, under
#             synthetic corruptions (blur, noise, thinning)
# By Oluwashina Oyeniran
# April 2, 2026
# ==============================================================================

# ------------------- 1. INSTALLATIONS & IMPORTS -------------------
!pip install -q torch torchvision pandas scipy matplotlib seaborn scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models
import numpy as np
import random
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             cohen_kappa_score, matthews_corrcoef,
                             precision_recall_fscore_support, confusion_matrix,
                             jaccard_score, log_loss, roc_curve, auc,
                             classification_report)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize
import pandas as pd
from google.colab import drive
from tqdm import tqdm
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

# ------------------- 2. EXTRACT DATASET FROM ZIP -------------------
if not os.path.exists("/content/YARS"):
    with zipfile.ZipFile('/content/YARS.zip', 'r') as zip_ref:
        zip_ref.extractall('/content')
    print("Dataset extracted to /content/YARS")
else:
    print("Dataset already exists at /content/YARS")
data_root = "/content/YARS"

# ------------------- 3. MOUNT DRIVE & CREATE OUTPUT FOLDER -------------------
drive.mount('/content/drive')
output_root = "/content/drive/MyDrive/TTA_Experiment_Results"
timestamp = time.strftime("%Y%m%d_%H%M%S")
output_folder = os.path.join(output_root, f"run_{timestamp}")
os.makedirs(output_folder, exist_ok=True)
print(f"All results will be saved to: {output_folder}")

# ------------------- 4. CONFIGURATION -------------------
IMG_SIZE = 224
NUM_CLASSES = 35
BATCH_SIZE = 128
NUM_EPOCHS_YORUBA = 60
NUM_EPOCHS_RESNET = 60
LR_YORUBA = 0.001
LR_RESNET = 0.001
USE_70_NEURONS = True
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# Synthetic corruptions
CORRUPTIONS = [
    {'name': 'Gaussian Blur (σ=2)', 'type': 'blur', 'intensity': 2.0},
    {'name': 'Salt-Pepper Noise (5%)', 'type': 'noise', 'intensity': 0.05},
    {'name': 'Thinning (threshold=0.3)', 'type': 'thinning', 'intensity': 0.3}
]

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ------------------- 5. DATA TRANSFORMS (temporary, for split only) -------------------
temp_transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

# ------------------- 6. STRATIFIED SPLIT FUNCTION -------------------
def stratified_split_from_single_folder(root_dir, train_ratio, val_ratio, test_ratio, seed=42):
    full_dataset = datasets.ImageFolder(root_dir, transform=temp_transform)
    class_indices = {cls: [] for cls in range(len(full_dataset.classes))}
    for idx, (_, label) in enumerate(full_dataset.samples):
        class_indices[label].append(idx)
    train_idx, val_idx, test_idx = [], [], []
    for cls, indices in class_indices.items():
        np.random.seed(seed)
        np.random.shuffle(indices)
        n_total = len(indices)
        n_train = int(train_ratio * n_total)
        n_val = int(val_ratio * n_total)
        n_test = n_total - n_train - n_val
        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train:n_train+n_val])
        test_idx.extend(indices[n_train+n_val:])
    return train_idx, val_idx, test_idx, full_dataset.classes

full_temp = datasets.ImageFolder(data_root, transform=temp_transform)
train_idx, val_idx, test_idx, class_names = stratified_split_from_single_folder(
    data_root, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed=42
)
print(f"Number of classes: {len(class_names)}")
print(f"Train samples: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")

# ------------------- 7. COMPUTE DATASET-SPECIFIC MEAN & STD (for YorubaNet) -------------------
temp_train_dataset = Subset(full_temp, train_idx)
temp_loader = DataLoader(temp_train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
def get_mean_std(loader):
    mean = 0.
    std = 0.
    total = 0
    for images, _ in loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total += batch_samples
    mean /= total
    std /= total
    return mean, std
dataset_mean, dataset_std = get_mean_std(temp_loader)
print(f"Dataset mean: {dataset_mean}, std: {dataset_std}")

# ------------------- 8. DEFINE TRANSFORMS FOR BOTH MODELS -------------------
# YorubaNet (dataset normalisation)
yoruba_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=dataset_mean, std=dataset_std)
])
yoruba_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=dataset_mean, std=dataset_std)
])

# ResNet-18 (pretrained on ImageNet) – uses ImageNet stats
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]
resnet_train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(5),
    transforms.RandomAffine(0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])
resnet_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# ------------------- 9. CREATE DATASETS & DATALOADERS -------------------
def make_datasets(root, indices, transform):
    full = datasets.ImageFolder(root, transform=transform)
    return Subset(full, indices)

# YorubaNet
yoruba_train_dataset = make_datasets(data_root, train_idx, yoruba_train_transform)
yoruba_val_dataset   = make_datasets(data_root, val_idx, yoruba_eval_transform)
yoruba_test_dataset  = make_datasets(data_root, test_idx, yoruba_eval_transform)

# ResNet-18
resnet_train_dataset = make_datasets(data_root, train_idx, resnet_train_transform)
resnet_val_dataset   = make_datasets(data_root, val_idx, resnet_eval_transform)
resnet_test_dataset  = make_datasets(data_root, test_idx, resnet_eval_transform)

# DataLoaders
yoruba_train_loader = DataLoader(yoruba_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
yoruba_val_loader   = DataLoader(yoruba_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
yoruba_test_loader  = DataLoader(yoruba_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

resnet_train_loader = DataLoader(resnet_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
resnet_val_loader   = DataLoader(resnet_val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
resnet_test_loader  = DataLoader(resnet_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ------------------- 10. YORUBANET ARCHITECTURE (NO DROPOUT) -------------------
class YorubaNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, use_70_neurons=True):
        super(YorubaNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1), nn.BatchNorm2d(8), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(8, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
        )
        if use_70_neurons:
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128*7*7, 70),
                nn.ReLU(inplace=True),
                nn.Linear(70, num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128*7*7, num_classes)
            )
    def forward(self, x):
        return self.classifier(self.features(x))

# ------------------- 11. TRAINING FUNCTION -------------------
def train_model(model, train_loader, val_loader, epochs, lr, device, model_name, output_folder):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} Train'):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
        train_loss = running_loss / len(train_loader)
        train_acc = 100 * correct / total
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, pred = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (pred == labels).sum().item()
        val_loss /= len(val_loader)
        val_acc = 100 * val_correct / val_total
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        scheduler.step(val_loss)
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(output_folder, f'best_{model_name}.pth'))
    model.load_state_dict(torch.load(os.path.join(output_folder, f'best_{model_name}.pth')))
    return model, history

# ------------------- 12. CORRUPTION FUNCTIONS (MODEL-AWARE NORMALISATION) -------------------
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(1,3,1,1).to(tensor.device)
    std = torch.tensor(std).view(1,3,1,1).to(tensor.device)
    return tensor * std + mean
def normalize(tensor, mean, std):
    mean = torch.tensor(mean).view(1,3,1,1).to(tensor.device)
    std = torch.tensor(std).view(1,3,1,1).to(tensor.device)
    return (tensor - mean) / std

def gaussian_blur(batch, mean, std, sigma=2.0):
    from torchvision.transforms.functional import gaussian_blur
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    blurred = torch.stack([gaussian_blur(img, kernel_size=5, sigma=sigma) for img in batch_denorm])
    return normalize(blurred, mean, std)

def salt_pepper_noise(batch, mean, std, prob=0.05):
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    noisy = batch_denorm.clone()
    mask = torch.rand_like(batch_denorm) < prob
    salt_pepper = torch.where(torch.rand_like(batch_denorm) < 0.5, torch.ones_like(batch_denorm), torch.zeros_like(batch_denorm))
    noisy[mask] = salt_pepper[mask]
    return normalize(noisy, mean, std)

def thinning(batch, mean, std, threshold=0.3):
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    thinned = batch_denorm.clone()
    thinned[thinned < threshold] = 0.0
    return normalize(thinned, mean, std)

def apply_corruption(batch, corruption_type, mean, std, intensity=None):
    if corruption_type == 'blur':
        return gaussian_blur(batch, mean, std, sigma=intensity if intensity else 2.0)
    elif corruption_type == 'noise':
        return salt_pepper_noise(batch, mean, std, prob=intensity if intensity else 0.05)
    elif corruption_type == 'thinning':
        return thinning(batch, mean, std, threshold=intensity if intensity else 0.3)
    else:
        return batch

# ------------------- 13. TEST-TIME ADAPTATION -------------------
def update_bn_stats(model, batch):
    model.train()
    with torch.no_grad():
        _ = model(batch)
    model.eval()

# ------------------- 14. METRICS AND EVALUATION -------------------
def compute_all_metrics(y_true, y_pred, y_proba=None, num_classes=NUM_CLASSES):
    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=range(num_classes), zero_division=0)
    precision_macro = precision_per_class.mean()
    recall_macro = recall_per_class.mean()
    f1_macro = f1_per_class.mean()
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    specificity_per_class = []
    for i in range(num_classes):
        tn = cm.sum() - (cm[i,:].sum() + cm[:,i].sum() - cm[i,i])
        fp = cm[:,i].sum() - cm[i,i]
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificity_per_class.append(spec)
    specificity_macro = np.mean(specificity_per_class)
    jaccard_per_class = jaccard_score(y_true, y_pred, average=None, labels=range(num_classes), zero_division=0)
    jaccard_macro = np.mean(jaccard_per_class)
    log_loss_value = None
    roc_auc_macro = None
    if y_proba is not None:
        log_loss_value = log_loss(y_true, y_proba, labels=range(num_classes))
        y_true_bin = label_binarize(y_true, classes=range(num_classes))
        try:
            aucs = []
            for i in range(num_classes):
                fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
                aucs.append(auc(fpr, tpr))
            roc_auc_macro = np.mean(aucs)
        except:
            pass
    return {
        'accuracy': accuracy, 'balanced_accuracy': balanced_acc, 'cohen_kappa': kappa, 'matthews_corrcoef': mcc,
        'precision_macro': precision_macro, 'recall_macro': recall_macro, 'f1_macro': f1_macro,
        'precision_weighted': precision_weighted, 'recall_weighted': recall_weighted, 'f1_weighted': f1_weighted,
        'specificity_macro': specificity_macro, 'jaccard_macro': jaccard_macro,
        'log_loss': log_loss_value, 'roc_auc_macro': roc_auc_macro,
        'per_class_precision': precision_per_class, 'per_class_recall': recall_per_class,
        'per_class_f1': f1_per_class, 'per_class_specificity': np.array(specificity_per_class),
        'per_class_jaccard': jaccard_per_class, 'confusion_matrix': cm
    }

def evaluate_detailed(model, test_loader, norm_mean, norm_std, corruption_type=None, intensity=None, use_tta=False, device='cuda'):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"Eval {corruption_type} TTA={use_tta}"):
        images, labels = images.to(device), labels.to(device)
        if corruption_type is not None:
            images = apply_corruption(images, corruption_type, norm_mean, norm_std, intensity)
        if use_tta:
            update_bn_stats(model, images)
        with torch.no_grad():
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    metrics = compute_all_metrics(all_labels, all_preds, all_probs, NUM_CLASSES)
    metrics['predictions'] = all_preds
    metrics['labels'] = all_labels
    metrics['probabilities'] = all_probs
    return metrics

def statistical_comparison(metrics_no, metrics_tta, labels):
    preds_no = metrics_no['predictions']
    preds_tta = metrics_tta['predictions']
    correct_no = (preds_no == labels).astype(int)
    correct_tta = (preds_tta == labels).astype(int)
    diff = correct_tta - correct_no
    t_stat, p_ttest = stats.ttest_rel(correct_tta, correct_no)
    w_stat, p_wilcox = stats.wilcoxon(correct_tta, correct_no)
    d = np.mean(diff) / (np.std(diff, ddof=1) + 1e-8)
    ci = stats.t.interval(0.95, len(diff)-1, loc=np.mean(diff), scale=stats.sem(diff))
    return {
        'improvement_accuracy': metrics_tta['accuracy'] - metrics_no['accuracy'],
        'improvement_balanced_acc': metrics_tta['balanced_accuracy'] - metrics_no['balanced_accuracy'],
        'improvement_f1_macro': metrics_tta['f1_macro'] - metrics_no['f1_macro'],
        'improvement_mcc': metrics_tta['matthews_corrcoef'] - metrics_no['matthews_corrcoef'],
        'improvement_jaccard': metrics_tta['jaccard_macro'] - metrics_no['jaccard_macro'],
        't_statistic': t_stat, 'p_value_ttest': p_ttest, 'wilcoxon_stat': w_stat,
        'p_value_wilcox': p_wilcox, 'cohens_d': d,
        'mean_improvement_points': np.mean(diff)*100,
        'ci_lower': ci[0]*100, 'ci_upper': ci[1]*100
    }

# ------------------- 15. TRAIN MODELS -------------------
print("\n" + "="*50)
print("Training YorubaNet (no dropout, dataset normalisation)")
print("="*50)
yorubanet = YorubaNet(num_classes=NUM_CLASSES, use_70_neurons=USE_70_NEURONS)
yorubanet, history_yoruba = train_model(yorubanet, yoruba_train_loader, yoruba_val_loader,
                                        epochs=NUM_EPOCHS_YORUBA, lr=LR_YORUBA,
                                        device=device, model_name='YorubaNet', output_folder=output_folder)

print("\n" + "="*50)
print("Fine-tuning ResNet-18 (pretrained, ImageNet normalisation)")
print("="*50)
resnet18 = models.resnet18(pretrained=True)
num_ftrs = resnet18.fc.in_features
resnet18.fc = nn.Linear(num_ftrs, NUM_CLASSES)
resnet18, history_resnet = train_model(resnet18, resnet_train_loader, resnet_val_loader,
                                       epochs=NUM_EPOCHS_RESNET, lr=LR_RESNET,
                                       device=device, model_name='ResNet18', output_folder=output_folder)

# ------------------- 16. EVALUATION ON CLEAN AND CORRUPTED SETS -------------------
def run_full_evaluation(model, test_loader, norm_mean, norm_std, corruptions, device):
    results = {}
    clean_res = evaluate_detailed(model, test_loader, norm_mean, norm_std, use_tta=False, device=device)
    results['clean'] = {'no_tta': clean_res}
    for corr in corruptions:
        res_no = evaluate_detailed(model, test_loader, norm_mean, norm_std,
                                   corruption_type=corr['type'], intensity=corr['intensity'],
                                   use_tta=False, device=device)
        res_tta = evaluate_detailed(model, test_loader, norm_mean, norm_std,
                                    corruption_type=corr['type'], intensity=corr['intensity'],
                                    use_tta=True, device=device)
        stats_dict = statistical_comparison(res_no, res_tta, res_no['labels'])
        results[corr['name']] = {'no_tta': res_no, 'tta': res_tta, 'stats': stats_dict}
    return results

dataset_mean_list = dataset_mean.cpu().numpy().tolist()
dataset_std_list = dataset_std.cpu().numpy().tolist()
imagenet_mean_list = imagenet_mean
imagenet_std_list = imagenet_std

print("\nEvaluating YorubaNet...")
results_yoruba = run_full_evaluation(yorubanet, yoruba_test_loader, dataset_mean_list, dataset_std_list, CORRUPTIONS, device)
print("\nEvaluating ResNet-18...")
results_resnet = run_full_evaluation(resnet18, resnet_test_loader, imagenet_mean_list, imagenet_std_list, CORRUPTIONS, device)

# ------------------- 17. SAVE RESULTS (CSV) -------------------
pd.DataFrame(history_yoruba).to_csv(os.path.join(output_folder, 'YorubaNet_training_history.csv'), index=False)
pd.DataFrame(history_resnet).to_csv(os.path.join(output_folder, 'ResNet18_training_history.csv'), index=False)

summary_rows = []
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    m_clean = res['clean']['no_tta']
    summary_rows.append({'Model': model_name, 'Corruption': 'Clean', 'TTA': 'No',
        'Accuracy': m_clean['accuracy'], 'Balanced_Acc': m_clean['balanced_accuracy'],
        'MCC': m_clean['matthews_corrcoef'], 'F1_macro': m_clean['f1_macro'],
        'Precision_macro': m_clean['precision_macro'], 'Recall_macro': m_clean['recall_macro'],
        'Specificity_macro': m_clean['specificity_macro'], 'Jaccard_macro': m_clean['jaccard_macro'],
        'LogLoss': m_clean['log_loss'], 'Cohen_Kappa': m_clean['cohen_kappa'], 'ROC_AUC_macro': m_clean['roc_auc_macro']})
    for corr_name, metrics in res.items():
        if corr_name == 'clean': continue
        m_no = metrics['no_tta']
        summary_rows.append({'Model': model_name, 'Corruption': corr_name, 'TTA': 'No',
            'Accuracy': m_no['accuracy'], 'Balanced_Acc': m_no['balanced_accuracy'],
            'MCC': m_no['matthews_corrcoef'], 'F1_macro': m_no['f1_macro'],
            'Precision_macro': m_no['precision_macro'], 'Recall_macro': m_no['recall_macro'],
            'Specificity_macro': m_no['specificity_macro'], 'Jaccard_macro': m_no['jaccard_macro'],
            'LogLoss': m_no['log_loss'], 'Cohen_Kappa': m_no['cohen_kappa'], 'ROC_AUC_macro': m_no['roc_auc_macro']})
        m_tta = metrics['tta']
        summary_rows.append({'Model': model_name, 'Corruption': corr_name, 'TTA': 'Yes',
            'Accuracy': m_tta['accuracy'], 'Balanced_Acc': m_tta['balanced_accuracy'],
            'MCC': m_tta['matthews_corrcoef'], 'F1_macro': m_tta['f1_macro'],
            'Precision_macro': m_tta['precision_macro'], 'Recall_macro': m_tta['recall_macro'],
            'Specificity_macro': m_tta['specificity_macro'], 'Jaccard_macro': m_tta['jaccard_macro'],
            'LogLoss': m_tta['log_loss'], 'Cohen_Kappa': m_tta['cohen_kappa'], 'ROC_AUC_macro': m_tta['roc_auc_macro']})
        s = metrics['stats']
        summary_rows.append({'Model': model_name, 'Corruption': corr_name, 'TTA': 'Improvement',
            'ΔAccuracy': s['improvement_accuracy'], 'ΔBalanced_Acc': s['improvement_balanced_acc'],
            'ΔMCC': s['improvement_mcc'], 'ΔF1_macro': s['improvement_f1_macro'],
            'ΔJaccard_macro': s['improvement_jaccard'], 'p-value (t-test)': s['p_value_ttest'],
            "Cohen's d": s['cohens_d'], '95% CI lower': s['ci_lower'], '95% CI upper': s['ci_upper']})
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv(os.path.join(output_folder, 'summary_metrics.csv'), index=False)

# Per-class metrics (shortened for brevity, same as earlier)
per_class_data = []
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    for corr_name, metrics in res.items():
        if corr_name == 'clean':
            m = metrics['no_tta']
            for i in range(NUM_CLASSES):
                per_class_data.append({'Model': model_name, 'Corruption': 'Clean', 'TTA': 'No', 'Class': class_names[i],
                    'Precision': m['per_class_precision'][i], 'Recall': m['per_class_recall'][i],
                    'F1': m['per_class_f1'][i], 'Specificity': m['per_class_specificity'][i],
                    'Jaccard': m['per_class_jaccard'][i]})
        else:
            m_no = metrics['no_tta']
            for i in range(NUM_CLASSES):
                per_class_data.append({'Model': model_name, 'Corruption': corr_name, 'TTA': 'No', 'Class': class_names[i],
                    'Precision': m_no['per_class_precision'][i], 'Recall': m_no['per_class_recall'][i],
                    'F1': m_no['per_class_f1'][i], 'Specificity': m_no['per_class_specificity'][i],
                    'Jaccard': m_no['per_class_jaccard'][i]})
            m_tta = metrics['tta']
            for i in range(NUM_CLASSES):
                per_class_data.append({'Model': model_name, 'Corruption': corr_name, 'TTA': 'Yes', 'Class': class_names[i],
                    'Precision': m_tta['per_class_precision'][i], 'Recall': m_tta['per_class_recall'][i],
                    'F1': m_tta['per_class_f1'][i], 'Specificity': m_tta['per_class_specificity'][i],
                    'Jaccard': m_tta['per_class_jaccard'][i]})
df_per_class = pd.DataFrame(per_class_data)
df_per_class.to_csv(os.path.join(output_folder, 'per_class_metrics.csv'), index=False)

# Confusion matrices
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    for corr_name, metrics in res.items():
        if corr_name != 'clean':
            cm_no = metrics['no_tta']['confusion_matrix']
            cm_tta = metrics['tta']['confusion_matrix']
            pd.DataFrame(cm_no).to_csv(os.path.join(output_folder, f'{model_name}_{corr_name}_cm_noTTA.csv'), index=False)
            pd.DataFrame(cm_tta).to_csv(os.path.join(output_folder, f'{model_name}_{corr_name}_cm_TTA.csv'), index=False)
            plt.figure(figsize=(12,10))
            sns.heatmap(cm_no, annot=False, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
            plt.title(f'{model_name} - {corr_name} without TTA'); plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'{model_name}_{corr_name}_cm_noTTA.png')); plt.close()
            plt.figure(figsize=(12,10))
            sns.heatmap(cm_tta, annot=False, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
            plt.title(f'{model_name} - {corr_name} with TTA'); plt.tight_layout()
            plt.savefig(os.path.join(output_folder, f'{model_name}_{corr_name}_cm_TTA.png')); plt.close()

# ------------------- 18. PLOTS -------------------
def plot_training_curves(history, model_name):
    epochs = range(1, len(history['train_acc'])+1)
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train Loss')
    plt.plot(epochs, history['val_loss'], 'r-', label='Val Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title(f'{model_name} - Loss'); plt.legend()
    plt.subplot(1,2,2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train Acc')
    plt.plot(epochs, history['val_acc'], 'r-', label='Val Acc')
    plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)'); plt.title(f'{model_name} - Accuracy'); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{model_name}_training_curves.png')); plt.close()
plot_training_curves(history_yoruba, 'YorubaNet')
plot_training_curves(history_resnet, 'ResNet18')

corruption_names = [c['name'] for c in CORRUPTIONS]
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    acc_no = [res['clean']['no_tta']['accuracy']] + [res[corr]['no_tta']['accuracy'] for corr in corruption_names]
    acc_yes = [res['clean']['no_tta']['accuracy']] + [res[corr]['tta']['accuracy'] for corr in corruption_names]
    x = np.arange(len(corruption_names)+1)
    width = 0.35
    plt.figure(figsize=(10,6))
    plt.bar(x - width/2, acc_no, width, label='Without TTA')
    plt.bar(x + width/2, acc_yes, width, label='With TTA')
    plt.xticks(x, ['Clean'] + corruption_names, rotation=45)
    plt.ylabel('Accuracy (%)'); plt.title(f'{model_name} - Accuracy with/without TTA'); plt.legend()
    plt.tight_layout(); plt.savefig(os.path.join(output_folder, f'{model_name}_accuracy_bars.png')); plt.close()

for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    improvements = [res[corr]['stats']['mean_improvement_points'] for corr in corruption_names]
    ci_lowers = [res[corr]['stats']['ci_lower'] for corr in corruption_names]
    ci_uppers = [res[corr]['stats']['ci_upper'] for corr in corruption_names]
    plt.figure(figsize=(10,6))
    plt.errorbar(corruption_names, improvements, yerr=[np.array(improvements)-np.array(ci_lowers), np.array(ci_uppers)-np.array(improvements)], fmt='o', capsize=5)
    plt.axhline(y=0, color='red', linestyle='--')
    plt.ylabel('Accuracy Improvement (%)'); plt.title(f'{model_name} - TTA Improvement with 95% CI')
    plt.xticks(rotation=45); plt.tight_layout(); plt.savefig(os.path.join(output_folder, f'{model_name}_improvement_CI.png')); plt.close()

corr_name = CORRUPTIONS[0]['name']
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    f1_no = res[corr_name]['no_tta']['per_class_f1']
    f1_tta = res[corr_name]['tta']['per_class_f1']
    improvement = f1_tta - f1_no
    plt.figure(figsize=(16,4))
    sns.heatmap(improvement.reshape(1,-1), annot=True, fmt='.3f', cmap='RdYlGn', center=0,
                xticklabels=class_names, yticklabels=[f'ΔF1\n{corr_name}'])
    plt.title(f'{model_name} - Per-class F1 improvement from TTA')
    plt.tight_layout(); plt.savefig(os.path.join(output_folder, f'{model_name}_per_class_f1_improvement.png')); plt.close()

def plot_roc_curves(y_true, y_proba, model_name, corruption_name, output_folder, class_names, num_classes=35):
    y_true_bin = label_binarize(y_true, classes=range(num_classes))
    plt.figure(figsize=(10,8))
    for i in range(min(10, num_classes)):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=1, label=f'{class_names[i]} (AUC={roc_auc:.2f})')
    plt.plot([0,1], [0,1], 'k--'); plt.xlim([0,1]); plt.ylim([0,1])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} - {corruption_name} ROC Curves (with TTA)')
    plt.legend(loc='lower right', fontsize=8); plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{model_name}_{corruption_name}_ROC_curves.png')); plt.close()
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    plot_roc_curves(res[corr_name]['tta']['labels'], res[corr_name]['tta']['probabilities'],
                    model_name, f"{corr_name}_with_TTA", output_folder, class_names, NUM_CLASSES)

def plot_calibration_curve(labels, probs, model_name, corruption_name, tta_status, output_folder):
    # Get predicted classes and their confidence (max probability)
    preds = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)
    # Binary correctness: 1 if correct, 0 otherwise
    correct = (preds == labels).astype(int)
    # Calibration curve expects binary labels (0/1) and predicted probabilities for the positive class
    fraction_pos, mean_pred = calibration_curve(correct, confidences, n_bins=10)
    plt.figure(figsize=(6,6))
    plt.plot(mean_pred, fraction_pos, 's-', label='Model')
    plt.plot([0,1], [0,1], 'k--', label='Perfect')
    plt.xlabel('Mean predicted probability (confidence)')
    plt.ylabel('Fraction of positives (accuracy)')
    plt.title(f'{model_name} - {corruption_name} {tta_status}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{model_name}_{corruption_name}_calibration_{tta_status}.png'))
    plt.close()

def visualize_sample_corruptions(test_loader, norm_mean, norm_std, device, output_folder, num_samples=5):
    images, labels = next(iter(test_loader))
    images = images[:num_samples].to(device)
    orig = images.clone()
    blurred = apply_corruption(images, 'blur', norm_mean, norm_std, intensity=2.0)
    noisy = apply_corruption(images, 'noise', norm_mean, norm_std, intensity=0.05)      # fixed
    thinned = apply_corruption(images, 'thinning', norm_mean, norm_std, intensity=0.3)  # fixed
    def show(tensor, mean, std):
        return denormalize(tensor, mean, std).cpu().detach()
    fig, axes = plt.subplots(num_samples, 4, figsize=(12, num_samples*2))
    for i in range(num_samples):
        axes[i,0].imshow(show(orig[i], norm_mean, norm_std).permute(1,2,0).numpy())
        axes[i,0].set_title('Original'); axes[i,0].axis('off')
        axes[i,1].imshow(show(blurred[i], norm_mean, norm_std).permute(1,2,0).numpy())
        axes[i,1].set_title('Blur (σ=2)'); axes[i,1].axis('off')
        axes[i,2].imshow(show(noisy[i], norm_mean, norm_std).permute(1,2,0).numpy())
        axes[i,2].set_title('Salt-Pepper (5%)'); axes[i,2].axis('off')
        axes[i,3].imshow(show(thinned[i], norm_mean, norm_std).permute(1,2,0).numpy())
        axes[i,3].set_title('Thinning (th=0.3)'); axes[i,3].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, 'sample_corruptions.png'))
    plt.close()

# ------------------- 19. FINAL SUMMARY -------------------
print("\n" + "="*60)
print(f"EXPERIMENT COMPLETED SUCCESSFULLY.")
print(f"All results saved to: {output_folder}")
print("Files generated: summary_metrics.csv, per_class_metrics.csv, training histories, confusion matrices, ROC curves, calibration plots, confidence histograms, sample corruptions, best model weights.")
print("="*60)
print("\n=== QUICK SUMMARY OF TTA IMPROVEMENT (Accuracy) ===")
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    print(f"\n{model_name}:")
    for corr in corruption_names:
        imp = res[corr]['stats']['improvement_accuracy']
        pval = res[corr]['stats']['p_value_ttest']
        print(f"  {corr}: +{imp:.2f}% (p={pval:.4f})")


Dataset extracted to /content/YARS
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All results will be saved to: /content/drive/MyDrive/TTA_Experiment_Results/run_20260404_203220
Using device: cuda
Number of classes: 35
Train samples: 8785, Val: 1890, Test: 1925
Dataset mean: tensor([0.9099, 0.9228, 0.9453]), std: tensor([0.1191, 0.1236, 0.1056])

Training YorubaNet (no dropout, dataset normalisation)


Epoch 1/60 Train: 100%|██████████| 69/69 [00:13<00:00,  5.14it/s]


Epoch 1: Train Loss=3.5794, Train Acc=2.86%, Val Loss=3.5572, Val Acc=2.86%


Epoch 2/60 Train: 100%|██████████| 69/69 [00:12<00:00,  5.36it/s]


Epoch 2: Train Loss=3.5567, Train Acc=2.99%, Val Loss=3.5567, Val Acc=2.86%


...

Epoch 5/60 Train: 100%|██████████| 69/69 [00:12<00:00,  5.31it/s]


Epoch 5: Train Loss=2.6095, Train Acc=21.91%, Val Loss=2.5394, Val Acc=21.75%


Epoch 6/60 Train:  45%|████▍     | 31/69 [00:05<00:07,  5.31it/s]
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_720/156769062.py", line 415, in <cell line: 0>
    yorubanet, history_yoruba = train_model(yorubanet, yoruba_train_loader, yoruba_val_loader,
                                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_720/156769062.py", line 228, in train_model
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} Train'):
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/tqdm/std.py", line 1181, in __iter__
    for obj in iterable:
               ^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 741, in __next__
    data = self._next_data()
           ^^^^^^^^^^^^^^^^^
  File "/usr

TypeError: object of type 'NoneType' has no len()

In [ ]:
# Install Graphviz (required for torchviz)
!apt-get install -y graphviz
!pip install torchviz

import torch
import torch.nn as nn
from torchviz import make_dot

# Define YorubaNet (same as your final version)
class YorubaNet(nn.Module):
    def __init__(self, num_classes=35, use_70_neurons=True):
        super(YorubaNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1), nn.BatchNorm2d(8), nn.ReLU(inplace=True), nn.MaxPool2d(2, stride=2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2, stride=2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, stride=2),
        )
        if use_70_neurons:
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128 * 7 * 7, 70),
                nn.ReLU(inplace=True),
                nn.Linear(70, num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(128 * 7 * 7, num_classes)
            )
    def forward(self, x):
        return self.classifier(self.features(x))

# Create model and a dummy input (batch size 1, 3 channels, 224x224)
model = YorubaNet()
dummy_input = torch.randn(1, 3, 224, 224)

# Generate graph and save as PNG
dot = make_dot(model(dummy_input), params=dict(model.named_parameters()))
dot.render('yorubanet_architecture', format='png', cleanup=True)
print("Architecture diagram saved as 'yorubanet_architecture.png'")

# Also print a text summary using torchsummary
!pip install torchsummary
from torchsummary import summary
summary(model, (3, 224, 224))

In [ ]:
# Simple summary without torchsummary
def model_summary(model, input_size=(3, 224, 224)):
    print(model)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTotal parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

model_summary(YorubaNet())

In [ ]:
# =============================================================================
# REGENERATE MISSING PLOTS (Calibration curves & confidence histograms)
# =============================================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, confusion_matrix
from google.colab import drive
from tqdm import tqdm
import time
import zipfile

# ------------------- 1. Mount Drive & Set Paths -------------------
drive.mount('/content/drive')
output_root = "/content/drive/MyDrive/TTA_Experiment_Results"
# Use the last successful run folder (you can change this if needed)
run_folder = "run_20260403_044533"   # <-- verify this is the correct folder name
output_folder = os.path.join(output_root, run_folder)
print(f"Output folder: {output_folder}")

# ------------------- 2. Extract Dataset if not already there -------------------
if not os.path.exists("/content/YARS"):
    with zipfile.ZipFile('/content/YARS.zip', 'r') as zip_ref:
        zip_ref.extractall('/content')
    print("Dataset extracted to /content/YARS")
data_root = "/content/YARS"

# ------------------- 3. Configuration (same as original) -------------------
IMG_SIZE = 224
NUM_CLASSES = 35
BATCH_SIZE = 128
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
CORRUPTIONS = [
    {'name': 'Gaussian Blur (σ=2)', 'type': 'blur', 'intensity': 2.0},
    {'name': 'Salt-Pepper Noise (5%)', 'type': 'noise', 'intensity': 0.05},
    {'name': 'Thinning (threshold=0.3)', 'type': 'thinning', 'intensity': 0.3}
]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ------------------- 4. Load dataset splits (recreate indices) -------------------
temp_transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])
full_temp = datasets.ImageFolder(data_root, transform=temp_transform)

def stratified_split_from_single_folder(root_dir, train_ratio, val_ratio, test_ratio, seed=42):
    full_dataset = datasets.ImageFolder(root_dir, transform=temp_transform)
    class_indices = {cls: [] for cls in range(len(full_dataset.classes))}
    for idx, (_, label) in enumerate(full_dataset.samples):
        class_indices[label].append(idx)
    train_idx, val_idx, test_idx = [], [], []
    for cls, indices in class_indices.items():
        np.random.seed(seed)
        np.random.shuffle(indices)
        n_total = len(indices)
        n_train = int(train_ratio * n_total)
        n_val = int(val_ratio * n_total)
        n_test = n_total - n_train - n_val
        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train:n_train+n_val])
        test_idx.extend(indices[n_train+n_val:])
    return train_idx, val_idx, test_idx, full_dataset.classes

train_idx, val_idx, test_idx, class_names = stratified_split_from_single_folder(
    data_root, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed=42
)
print(f"Test samples: {len(test_idx)}")

# ------------------- 5. Define transforms (dataset-specific for YorubaNet, ImageNet for ResNet) -------------------
# Compute dataset mean/std (same as before)
temp_train_dataset = Subset(full_temp, train_idx)
temp_loader = DataLoader(temp_train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
def get_mean_std(loader):
    mean = 0.; std = 0.; total = 0
    for images, _ in loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total += batch_samples
    mean /= total; std /= total
    return mean, std
dataset_mean, dataset_std = get_mean_std(temp_loader)
dataset_mean_list = dataset_mean.cpu().numpy().tolist()
dataset_std_list = dataset_std.cpu().numpy().tolist()

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Transforms for YorubaNet
yoruba_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=dataset_mean_list, std=dataset_std_list)
])

# Transforms for ResNet-18
resnet_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# ------------------- 6. Create test datasets and loaders -------------------
def make_test_dataset(root, indices, transform):
    full = datasets.ImageFolder(root, transform=transform)
    return Subset(full, indices)

yoruba_test_dataset = make_test_dataset(data_root, test_idx, yoruba_eval_transform)
resnet_test_dataset = make_test_dataset(data_root, test_idx, resnet_eval_transform)

yoruba_test_loader = DataLoader(yoruba_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
resnet_test_loader = DataLoader(resnet_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# ------------------- 7. Define models and load saved weights -------------------
class YorubaNet(nn.Module):
    def __init__(self, num_classes=35, use_70_neurons=True):
        super(YorubaNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,8,3,padding=1), nn.BatchNorm2d(8), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(8,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
        )
        if use_70_neurons:
            self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128*7*7,70), nn.ReLU(inplace=True), nn.Linear(70,num_classes))
        else:
            self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128*7*7,num_classes))
    def forward(self, x): return self.classifier(self.features(x))

# Load YorubaNet
yorubanet = YorubaNet(num_classes=NUM_CLASSES, use_70_neurons=True)
yorubanet.load_state_dict(torch.load(os.path.join(output_folder, 'best_YorubaNet.pth'), map_location=device))
yorubanet = yorubanet.to(device)
print("YorubaNet loaded.")

# Load ResNet-18
resnet18 = models.resnet18(pretrained=False)
resnet18.fc = nn.Linear(512, NUM_CLASSES)
resnet18.load_state_dict(torch.load(os.path.join(output_folder, 'best_ResNet18.pth'), map_location=device))
resnet18 = resnet18.to(device)
print("ResNet-18 loaded.")

# ------------------- 8. Reuse evaluation functions (simplified) -------------------
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(1,3,1,1).to(tensor.device)
    std = torch.tensor(std).view(1,3,1,1).to(tensor.device)
    return tensor * std + mean

def normalize(tensor, mean, std):
    mean = torch.tensor(mean).view(1,3,1,1).to(tensor.device)
    std = torch.tensor(std).view(1,3,1,1).to(tensor.device)
    return (tensor - mean) / std

def gaussian_blur(batch, mean, std, sigma=2.0):
    from torchvision.transforms.functional import gaussian_blur
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    blurred = torch.stack([gaussian_blur(img, 5, sigma=sigma) for img in batch_denorm])
    return normalize(blurred, mean, std)

def salt_pepper_noise(batch, mean, std, prob=0.05):
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    noisy = batch_denorm.clone()
    mask = torch.rand_like(batch_denorm) < prob
    salt_pepper = torch.where(torch.rand_like(batch_denorm) < 0.5, torch.ones_like(batch_denorm), torch.zeros_like(batch_denorm))
    noisy[mask] = salt_pepper[mask]
    return normalize(noisy, mean, std)

def thinning(batch, mean, std, threshold=0.3):
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    thinned = batch_denorm.clone()
    thinned[thinned < threshold] = 0.0
    return normalize(thinned, mean, std)

def apply_corruption(batch, corruption_type, mean, std, intensity=None):
    if corruption_type == 'blur':
        return gaussian_blur(batch, mean, std, sigma=intensity or 2.0)
    elif corruption_type == 'noise':
        return salt_pepper_noise(batch, mean, std, prob=intensity or 0.05)
    elif corruption_type == 'thinning':
        return thinning(batch, mean, std, threshold=intensity or 0.3)
    else:
        return batch

def update_bn_stats(model, batch):
    model.train()
    with torch.no_grad():
        _ = model(batch)
    model.eval()

def evaluate_detailed(model, test_loader, norm_mean, norm_std, corruption_type=None, intensity=None, use_tta=False, device='cuda'):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"Eval {corruption_type} TTA={use_tta}"):
        images, labels = images.to(device), labels.to(device)
        if corruption_type is not None:
            images = apply_corruption(images, corruption_type, norm_mean, norm_std, intensity)
        if use_tta:
            update_bn_stats(model, images)
        with torch.no_grad():
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    return {'predictions': all_preds, 'labels': all_labels, 'probabilities': all_probs}

# ------------------- 9. Run evaluations for both models (only needed for calibration plots) -------------------
corruption_names = [c['name'] for c in CORRUPTIONS]

# YorubaNet
results_yoruba = {}
clean_res = evaluate_detailed(yorubanet, yoruba_test_loader, dataset_mean_list, dataset_std_list, use_tta=False, device=device)
results_yoruba['clean'] = {'no_tta': clean_res}
for corr in CORRUPTIONS:
    res_no = evaluate_detailed(yorubanet, yoruba_test_loader, dataset_mean_list, dataset_std_list,
                               corruption_type=corr['type'], intensity=corr['intensity'], use_tta=False, device=device)
    res_tta = evaluate_detailed(yorubanet, yoruba_test_loader, dataset_mean_list, dataset_std_list,
                                corruption_type=corr['type'], intensity=corr['intensity'], use_tta=True, device=device)
    results_yoruba[corr['name']] = {'no_tta': res_no, 'tta': res_tta}

# ResNet-18
results_resnet = {}
clean_res = evaluate_detailed(resnet18, resnet_test_loader, imagenet_mean, imagenet_std, use_tta=False, device=device)
results_resnet['clean'] = {'no_tta': clean_res}
for corr in CORRUPTIONS:
    res_no = evaluate_detailed(resnet18, resnet_test_loader, imagenet_mean, imagenet_std,
                               corruption_type=corr['type'], intensity=corr['intensity'], use_tta=False, device=device)
    res_tta = evaluate_detailed(resnet18, resnet_test_loader, imagenet_mean, imagenet_std,
                                corruption_type=corr['type'], intensity=corr['intensity'], use_tta=True, device=device)
    results_resnet[corr['name']] = {'no_tta': res_no, 'tta': res_tta}

# ------------------- 10. Generate calibration curves and confidence histograms -------------------
def plot_calibration_curve(labels, probs, model_name, corruption_name, tta_status, output_folder):
    preds = np.argmax(probs, axis=1)
    confidences = np.max(probs, axis=1)
    correct = (preds == labels).astype(int)
    fraction_pos, mean_pred = calibration_curve(correct, confidences, n_bins=10)
    plt.figure(figsize=(6,6))
    plt.plot(mean_pred, fraction_pos, 's-', label='Model')
    plt.plot([0,1], [0,1], 'k--', label='Perfect')
    plt.xlabel('Mean predicted probability (confidence)')
    plt.ylabel('Fraction of positives (accuracy)')
    plt.title(f'{model_name} - {corruption_name} {tta_status}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{model_name}_{corruption_name}_calibration_{tta_status}.png'))
    plt.close()
    print(f"Saved: {model_name}_{corruption_name}_calibration_{tta_status}.png")

# Calibration curves
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    for corr in corruption_names:
        plot_calibration_curve(res[corr]['no_tta']['labels'], res[corr]['no_tta']['probabilities'],
                               model_name, corr, 'without_TTA', output_folder)
        plot_calibration_curve(res[corr]['tta']['labels'], res[corr]['tta']['probabilities'],
                               model_name, corr, 'with_TTA', output_folder)

# Confidence histograms
for model_name, res in [('YorubaNet', results_yoruba), ('ResNet18', results_resnet)]:
    for corr in corruption_names:
        conf_no = np.max(res[corr]['no_tta']['probabilities'], axis=1)
        conf_tta = np.max(res[corr]['tta']['probabilities'], axis=1)
        plt.figure(figsize=(10,5))
        plt.hist(conf_no, bins=50, alpha=0.5, label='Without TTA', density=True)
        plt.hist(conf_tta, bins=50, alpha=0.5, label='With TTA', density=True)
        plt.xlabel('Confidence'); plt.ylabel('Density')
        plt.title(f'{model_name} - {corr} Confidence Distribution')
        plt.legend(); plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f'{model_name}_{corr}_confidence_dist.png'))
        plt.close()
        print(f"Saved: {model_name}_{corr}_confidence_dist.png")

print("\n=== All missing plots regenerated ===")

In [4]:
# =============================================================================
# COMPLETE COMPARATIVE STUDY: BN‑update vs TENT vs TTT (with full ablations)
# For low‑resource handwritten Yoruba character recognition
# Loads pre‑trained models from previous experiment
# =============================================================================

# ------------------- 1. INSTALLATIONS & IMPORTS -------------------
!pip install -q torch torchvision pandas scipy matplotlib seaborn scikit-learn

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models
import numpy as np
import random
import os
import copy
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import accuracy_score, balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, jaccard_score, log_loss, roc_curve, auc
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize
import pandas as pd
from google.colab import drive
from tqdm import tqdm
import time
import zipfile
import warnings
warnings.filterwarnings('ignore')

# ------------------- 2. MOUNT DRIVE & SET PATHS -------------------
drive.mount('/content/drive')

# *** CHANGE THIS TO YOUR PREVIOUS RUN FOLDER CONTAINING best_*.pth ***
prev_run_folder = "/content/drive/MyDrive/TTA_Experiment_Results/run_20260403_044533"  # Example

output_root = "/content/drive/MyDrive/TTA_Comparative_Ablation"
timestamp = time.strftime("%Y%m%d_%H%M%S")
output_folder = os.path.join(output_root, f"ablation_{timestamp}")
os.makedirs(output_folder, exist_ok=True)
print(f"Ablation results will be saved to: {output_folder}")

# ------------------- 3. CONFIGURATION (same as original) -------------------
IMG_SIZE = 224
NUM_CLASSES = 35
BATCH_SIZE = 128
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
CORRUPTIONS = [
    {'name': 'Gaussian Blur (σ=2)', 'type': 'blur', 'intensity': 2.0},
    {'name': 'Salt-Pepper Noise (5%)', 'type': 'noise', 'intensity': 0.05},
    {'name': 'Thinning (threshold=0.3)', 'type': 'thinning', 'intensity': 0.3}
]
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ------------------- 4. LOAD DATASET AND CREATE TEST LOADER -------------------
data_root = "/content/YARS"   # Assumes dataset is already extracted
if not os.path.exists(data_root):
    with zipfile.ZipFile('/content/YARS.zip', 'r') as zip_ref:
        zip_ref.extractall('/content')
    print("Dataset extracted to /content/YARS")

# Helper function to get stratified split indices (same as original)
def stratified_split_from_single_folder(root_dir, train_ratio, val_ratio, test_ratio, seed=42):
    temp_transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])
    full_dataset = datasets.ImageFolder(root_dir, transform=temp_transform)
    class_indices = {cls: [] for cls in range(len(full_dataset.classes))}
    for idx, (_, label) in enumerate(full_dataset.samples):
        class_indices[label].append(idx)
    train_idx, val_idx, test_idx = [], [], []
    for cls, indices in class_indices.items():
        np.random.seed(seed)
        np.random.shuffle(indices)
        n_total = len(indices)
        n_train = int(train_ratio * n_total)
        n_val = int(val_ratio * n_total)
        n_test = n_total - n_train - n_val
        train_idx.extend(indices[:n_train])
        val_idx.extend(indices[n_train:n_train+n_val])
        test_idx.extend(indices[n_train+n_val:])
    return train_idx, val_idx, test_idx, full_dataset.classes

_, _, test_idx, class_names = stratified_split_from_single_folder(data_root, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed=42)

# Dataset statistics (from previous run)
dataset_mean = [0.9099, 0.9228, 0.9453]
dataset_std = [0.1191, 0.1236, 0.1056]
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Transforms
yoruba_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=dataset_mean, std=dataset_std)
])
resnet_eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# Create test datasets
yoruba_test_dataset = Subset(datasets.ImageFolder(data_root, transform=yoruba_eval_transform), test_idx)
resnet_test_dataset = Subset(datasets.ImageFolder(data_root, transform=resnet_eval_transform), test_idx)

yoruba_test_loader = DataLoader(yoruba_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
resnet_test_loader = DataLoader(resnet_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Also create a training loader for TTT's rotation head pre‑training (use training indices)
full_temp = datasets.ImageFolder(data_root, transform=transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()]))
train_idx, _, _, _ = stratified_split_from_single_folder(data_root, TRAIN_RATIO, VAL_RATIO, TEST_RATIO, seed=42)
train_dataset = Subset(full_temp, train_idx)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

# ------------------- 5. DEFINE MODELS (same architectures) -------------------
class YorubaNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, use_70_neurons=True):
        super(YorubaNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,8,3,padding=1), nn.BatchNorm2d(8), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(8,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2,2),
        )
        if use_70_neurons:
            self.classifier = nn.Sequential(
                nn.Flatten(), nn.Linear(128*7*7,70), nn.ReLU(inplace=True), nn.Linear(70,num_classes)
            )
        else:
            self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(128*7*7,num_classes))
    def forward(self, x):
        return self.classifier(self.features(x))

# For ResNet, we create a wrapper that can return features for rotation head
class ResNetWithFeatures(nn.Module):
    def __init__(self, original):
        super(ResNetWithFeatures, self).__init__()
        # Copy all layers from the original ResNet
        self.conv1 = original.conv1
        self.bn1 = original.bn1
        self.relu = original.relu
        self.maxpool = original.maxpool
        self.layer1 = original.layer1
        self.layer2 = original.layer2
        self.layer3 = original.layer3
        self.layer4 = original.layer4
        self.avgpool = original.avgpool
        self.fc = original.fc  # keep for main task
        # Add rotation head later
        self.rot_head = None

    def forward_features(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        return x.view(x.size(0), -1)

    def forward(self, x):
        return self.fc(self.forward_features(x))

# ------------------- 6. LOAD PRE‑TRAINED MODELS -------------------
print("Loading pre‑trained models...")
yorubanet = YorubaNet(num_classes=NUM_CLASSES, use_70_neurons=True)
yorubanet.load_state_dict(torch.load(os.path.join(prev_run_folder, 'best_YorubaNet.pth'), map_location=device))
yorubanet = yorubanet.to(device)
print("YorubaNet loaded.")

# Load original ResNet and wrap it
resnet_original = models.resnet18(pretrained=False)
resnet_original.fc = nn.Linear(512, NUM_CLASSES)
resnet_original.load_state_dict(torch.load(os.path.join(prev_run_folder, 'best_ResNet18.pth'), map_location=device))
resnet18 = ResNetWithFeatures(resnet_original)
resnet18 = resnet18.to(device)
print("ResNet-18 loaded.")

# ------------------- 7. CORRUPTION FUNCTIONS (same as original) -------------------
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).view(1,3,1,1).to(tensor.device)
    std = torch.tensor(std).view(1,3,1,1).to(tensor.device)
    return tensor * std + mean
def normalize(tensor, mean, std):
    mean = torch.tensor(mean).view(1,3,1,1).to(tensor.device)
    std = torch.tensor(std).view(1,3,1,1).to(tensor.device)
    return (tensor - mean) / std

def gaussian_blur(batch, mean, std, sigma=2.0):
    from torchvision.transforms.functional import gaussian_blur
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    blurred = torch.stack([gaussian_blur(img, kernel_size=5, sigma=sigma) for img in batch_denorm])
    return normalize(blurred, mean, std)

def salt_pepper_noise(batch, mean, std, prob=0.05):
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    noisy = batch_denorm.clone()
    mask = torch.rand_like(batch_denorm) < prob
    salt_pepper = torch.where(torch.rand_like(batch_denorm) < 0.5, torch.ones_like(batch_denorm), torch.zeros_like(batch_denorm))
    noisy[mask] = salt_pepper[mask]
    return normalize(noisy, mean, std)

def thinning(batch, mean, std, threshold=0.3):
    batch_denorm = denormalize(batch, mean, std)
    batch_denorm = torch.clamp(batch_denorm, 0, 1)
    thinned = batch_denorm.clone()
    thinned[thinned < threshold] = 0.0
    return normalize(thinned, mean, std)

def apply_corruption(batch, corruption_type, mean, std, intensity=None):
    if corruption_type == 'blur':
        return gaussian_blur(batch, mean, std, sigma=intensity if intensity else 2.0)
    elif corruption_type == 'noise':
        return salt_pepper_noise(batch, mean, std, prob=intensity if intensity else 0.05)
    elif corruption_type == 'thinning':
        return thinning(batch, mean, std, threshold=intensity if intensity else 0.3)
    else:
        return batch

# ------------------- 8. METRICS FUNCTION (same) -------------------
def compute_all_metrics(y_true, y_pred, y_proba=None, num_classes=NUM_CLASSES):
    accuracy = accuracy_score(y_true, y_pred)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=range(num_classes), zero_division=0)
    precision_macro = precision_per_class.mean()
    recall_macro = recall_per_class.mean()
    f1_macro = f1_per_class.mean()
    specificity_per_class = []
    cm = confusion_matrix(y_true, y_pred, labels=range(num_classes))
    for i in range(num_classes):
        tn = cm.sum() - (cm[i,:].sum() + cm[:,i].sum() - cm[i,i])
        fp = cm[:,i].sum() - cm[i,i]
        spec = tn / (tn + fp) if (tn+fp)>0 else 0
        specificity_per_class.append(spec)
    specificity_macro = np.mean(specificity_per_class)
    jaccard_per_class = jaccard_score(y_true, y_pred, average=None, labels=range(num_classes), zero_division=0)
    jaccard_macro = np.mean(jaccard_per_class)
    log_loss_value = None
    roc_auc_macro = None
    if y_proba is not None:
        log_loss_value = log_loss(y_true, y_proba, labels=range(num_classes))
        y_true_bin = label_binarize(y_true, classes=range(num_classes))
        try:
            aucs = []
            for i in range(num_classes):
                fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
                aucs.append(auc(fpr, tpr))
            roc_auc_macro = np.mean(aucs)
        except:
            pass
    return {'accuracy': accuracy, 'balanced_accuracy': balanced_acc, 'cohen_kappa': kappa, 'matthews_corrcoef': mcc,
            'precision_macro': precision_macro, 'recall_macro': recall_macro, 'f1_macro': f1_macro,
            'specificity_macro': specificity_macro, 'jaccard_macro': jaccard_macro,
            'log_loss': log_loss_value, 'roc_auc_macro': roc_auc_macro}

# ------------------- 9. BASELINE (NO TTA) -------------------
def evaluate_baseline(model, test_loader, norm_mean, norm_std, corruption_type=None, intensity=None, device='cuda'):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"Baseline {corruption_type}"):
        images, labels = images.to(device), labels.to(device)
        if corruption_type is not None:
            images = apply_corruption(images, corruption_type, norm_mean, norm_std, intensity)
        with torch.no_grad():
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    metrics = compute_all_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs), NUM_CLASSES)
    return metrics

# ------------------- 10. BN‑UPDATE TTA (original method) -------------------
def update_bn_stats(model, batch):
    model.train()
    with torch.no_grad():
        _ = model(batch)
    model.eval()

def evaluate_bn_tta(model, test_loader, norm_mean, norm_std, corruption_type=None, intensity=None, device='cuda'):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"BN‑TTA {corruption_type}"):
        images, labels = images.to(device), labels.to(device)
        if corruption_type is not None:
            images = apply_corruption(images, corruption_type, norm_mean, norm_std, intensity)
        update_bn_stats(model, images)
        with torch.no_grad():
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    metrics = compute_all_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs), NUM_CLASSES)
    return metrics

# ------------------- 11. TENT (Test‑Time Entropy Minimization) - CORRECTED -------------------
def evaluate_tent(model, test_loader, norm_mean, norm_std, corruption_type=None, intensity=None,
                  lr=1e-3, steps=1, device='cuda'):
    model_tent = copy.deepcopy(model)
    model_tent.train()

    # First, set all parameters to not require gradients
    for param in model_tent.parameters():
        param.requires_grad = False

    # Then, for every BatchNorm2d module, set its weight and bias to require gradients
    for module in model_tent.modules():
        if isinstance(module, nn.BatchNorm2d):
            if module.weight is not None:
                module.weight.requires_grad = True
            if module.bias is not None:
                module.bias.requires_grad = True

    # Collect trainable parameters
    trainable_params = [p for p in model_tent.parameters() if p.requires_grad]
    if len(trainable_params) == 0:
        raise ValueError("No trainable parameters found for TENT. Check BN layer identification.")

    optimizer = optim.Adam(trainable_params, lr=lr)

    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"TENT {corruption_type} (lr={lr}, steps={steps})"):
        images, labels = images.to(device), labels.to(device)
        if corruption_type is not None:
            images = apply_corruption(images, corruption_type, norm_mean, norm_std, intensity)

        # Adaptation steps
        for _ in range(steps):
            outputs = model_tent(images)
            probs = torch.softmax(outputs, dim=1)
            entropy = -torch.sum(probs * torch.log(probs + 1e-8), dim=1).mean()
            optimizer.zero_grad()
            entropy.backward()
            optimizer.step()

        # Final prediction
        with torch.no_grad():
            outputs = model_tent(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    metrics = compute_all_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs), NUM_CLASSES)
    return metrics

# ------------------- 12. TTT (Test‑Time Training with Rotation Prediction) -------------------
def add_rotation_head(model, num_rotations=4):
    """Add a rotation prediction head on top of the feature extractor."""
    if isinstance(model, YorubaNet):
        feat_dim = 128 * 7 * 7
        model.rot_head = nn.Linear(feat_dim, num_rotations).to(device)
    elif isinstance(model, ResNetWithFeatures):
        feat_dim = 512  # after global average pooling
        model.rot_head = nn.Linear(feat_dim, num_rotations).to(device)
    else:
        raise TypeError("Unsupported model type for TTT")
    return model

def train_rotation_head(model, train_loader, epochs=5, lr=1e-3, device='cuda'):
    """Pre‑train the rotation head on the training set using rotation self‑supervision."""
    model.train()
    # Freeze base model, only train rot_head
    for param in model.parameters():
        param.requires_grad = False
    for param in model.rot_head.parameters():
        param.requires_grad = True
    optimizer = optim.Adam(model.rot_head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        total_loss = 0
        correct = 0
        total = 0
        for images, _ in tqdm(train_loader, desc=f"Pre‑train rotation head epoch {epoch+1}"):
            images = images.to(device)
            # Apply random rotations (0, 90, 180, 270 degrees)
            rot_labels = torch.randint(0, 4, (images.size(0),), device=device)
            rot_images = []
            for i, rot in enumerate(rot_labels):
                if rot == 0:
                    rot_img = images[i]
                elif rot == 1:
                    rot_img = torch.rot90(images[i], 1, [1,2])
                elif rot == 2:
                    rot_img = torch.rot90(images[i], 2, [1,2])
                else:
                    rot_img = torch.rot90(images[i], 3, [1,2])
                rot_images.append(rot_img)
            rot_images = torch.stack(rot_images)
            # Forward through base model to get features
            if isinstance(model, YorubaNet):
                features = model.features(rot_images)
                features = features.view(features.size(0), -1)  # flatten
            elif isinstance(model, ResNetWithFeatures):
                features = model.forward_features(rot_images)
            else:
                raise TypeError("Unsupported model type")
            outputs = model.rot_head(features)
            loss = criterion(outputs, rot_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            correct += (pred == rot_labels).sum().item()
            total += rot_labels.size(0)
        print(f"Epoch {epoch+1}: Loss={total_loss/len(train_loader):.4f}, Acc={100*correct/total:.2f}%")
    # Re-enable base model gradients for test‑time fine‑tuning
    for param in model.parameters():
        param.requires_grad = True
    return model

def evaluate_ttt(model, test_loader, norm_mean, norm_std, corruption_type=None, intensity=None,
                 lr=1e-4, steps=5, device='cuda'):
    """Test‑time training: update the whole model using rotation prediction loss."""
    model_ttt = copy.deepcopy(model)
    model_ttt.train()
    optimizer = optim.Adam(model_ttt.parameters(), lr=lr)
    criterion_rot = nn.CrossEntropyLoss()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"TTT {corruption_type}"):
        images, labels = images.to(device), labels.to(device)
        if corruption_type is not None:
            images = apply_corruption(images, corruption_type, norm_mean, norm_std, intensity)
        # Perform several gradient steps on rotation prediction using the same images (no labels)
        for _ in range(steps):
            # Generate random rotations for each image in the batch
            rot_labels = torch.randint(0, 4, (images.size(0),), device=device)
            rot_images = []
            for i, rot in enumerate(rot_labels):
                if rot == 0:
                    rot_img = images[i]
                elif rot == 1:
                    rot_img = torch.rot90(images[i], 1, [1,2])
                elif rot == 2:
                    rot_img = torch.rot90(images[i], 2, [1,2])
                else:
                    rot_img = torch.rot90(images[i], 3, [1,2])
                rot_images.append(rot_img)
            rot_images = torch.stack(rot_images)
            # Forward pass to get features and rotation logits
            if isinstance(model_ttt, YorubaNet):
                features = model_ttt.features(rot_images)
                features = features.view(features.size(0), -1)
                rot_logits = model_ttt.rot_head(features)
            elif isinstance(model_ttt, ResNetWithFeatures):
                features = model_ttt.forward_features(rot_images)
                rot_logits = model_ttt.rot_head(features)
            else:
                raise TypeError("Unsupported model type")
            loss_rot = criterion_rot(rot_logits, rot_labels)
            optimizer.zero_grad()
            loss_rot.backward()
            optimizer.step()
        # Now make main task prediction
        with torch.no_grad():
            outputs = model_ttt(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    metrics = compute_all_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs), NUM_CLASSES)
    return metrics

# ------------------- 13. RUN EVALUATIONS FOR ALL METHODS -------------------
results = []  # list of dicts

# Prepare models for TTT (add rotation heads and pre‑train)
print("\nPre‑training rotation heads for TTT...")
yorubanet_ttt = add_rotation_head(copy.deepcopy(yorubanet))
train_rotation_head(yorubanet_ttt, train_loader, epochs=5, lr=1e-3, device=device)

resnet18_ttt = add_rotation_head(copy.deepcopy(resnet18))
train_rotation_head(resnet18_ttt, train_loader, epochs=5, lr=1e-3, device=device)

models_to_test = [
    ('YorubaNet', yorubanet, yoruba_test_loader, dataset_mean, dataset_std),
    ('ResNet18', resnet18, resnet_test_loader, imagenet_mean, imagenet_std)
]

# Run evaluations
for model_name, model, loader, mean, std in models_to_test:
    for corr in CORRUPTIONS:
        corr_name = corr['name']
        corr_type = corr['type']
        intensity = corr['intensity']

        # Baseline
        print(f"\nRunning baseline: {model_name}, {corr_name}")
        m_baseline = evaluate_baseline(model, loader, mean, std, corr_type, intensity, device)
        results.append({
            'Model': model_name, 'Corruption': corr_name, 'Method': 'No TTA',
            'Accuracy': m_baseline['accuracy'], 'Balanced_Acc': m_baseline['balanced_accuracy'],
            'MCC': m_baseline['matthews_corrcoef'], 'F1_macro': m_baseline['f1_macro'],
            'Specificity': m_baseline['specificity_macro'], 'Jaccard': m_baseline['jaccard_macro'],
            'LogLoss': m_baseline['log_loss'], 'ROC_AUC': m_baseline['roc_auc_macro']
        })

        # BN‑TTA
        print(f"Running BN‑TTA: {model_name}, {corr_name}")
        m_bn = evaluate_bn_tta(model, loader, mean, std, corr_type, intensity, device)
        results.append({
            'Model': model_name, 'Corruption': corr_name, 'Method': 'BN‑TTA',
            'Accuracy': m_bn['accuracy'], 'Balanced_Acc': m_bn['balanced_accuracy'],
            'MCC': m_bn['matthews_corrcoef'], 'F1_macro': m_bn['f1_macro'],
            'Specificity': m_bn['specificity_macro'], 'Jaccard': m_bn['jaccard_macro'],
            'LogLoss': m_bn['log_loss'], 'ROC_AUC': m_bn['roc_auc_macro']
        })

        # TENT (default lr=1e-3, steps=1)
        print(f"Running TENT: {model_name}, {corr_name}")
        m_tent = evaluate_tent(model, loader, mean, std, corr_type, intensity, lr=1e-3, steps=1, device=device)
        results.append({
            'Model': model_name, 'Corruption': corr_name, 'Method': 'TENT',
            'Accuracy': m_tent['accuracy'], 'Balanced_Acc': m_tent['balanced_accuracy'],
            'MCC': m_tent['matthews_corrcoef'], 'F1_macro': m_tent['f1_macro'],
            'Specificity': m_tent['specificity_macro'], 'Jaccard': m_tent['jaccard_macro'],
            'LogLoss': m_tent['log_loss'], 'ROC_AUC': m_tent['roc_auc_macro']
        })

        # TTT (using pre‑trained rotation head)
        print(f"Running TTT: {model_name}, {corr_name}")
        if model_name == 'YorubaNet':
            model_ttt = yorubanet_ttt
        else:
            model_ttt = resnet18_ttt
        m_ttt = evaluate_ttt(model_ttt, loader, mean, std, corr_type, intensity, lr=1e-4, steps=3, device=device)
        results.append({
            'Model': model_name, 'Corruption': corr_name, 'Method': 'TTT',
            'Accuracy': m_ttt['accuracy'], 'Balanced_Acc': m_ttt['balanced_accuracy'],
            'MCC': m_ttt['matthews_corrcoef'], 'F1_macro': m_ttt['f1_macro'],
            'Specificity': m_ttt['specificity_macro'], 'Jaccard': m_ttt['jaccard_macro'],
            'LogLoss': m_ttt['log_loss'], 'ROC_AUC': m_ttt['roc_auc_macro']
        })

# ------------------- 14. ABLATION STUDIES (for all corruptions) -------------------
ablation_results = []

# Ablation 1: TENT learning rate (1e-4, 1e-3, 1e-2) for all corruptions
for model_name, model, loader, mean, std in models_to_test:
    for lr in [1e-4, 1e-3, 1e-2]:
        for corr in CORRUPTIONS:
            print(f"Ablation TENT lr={lr}: {model_name}, {corr['name']}")
            m = evaluate_tent(model, loader, mean, std, corr['type'], corr['intensity'], lr=lr, steps=1, device=device)
            ablation_results.append({
                'Model': model_name, 'Corruption': corr['name'], 'Method': f'TENT_lr={lr}',
                'Accuracy': m['accuracy'], 'Balanced_Acc': m['balanced_accuracy'],
                'MCC': m['matthews_corrcoef'], 'F1_macro': m['f1_macro']
            })

# Ablation 2: TENT number of update steps (1, 3, 5) for all corruptions
for model_name, model, loader, mean, std in models_to_test:
    for steps in [1, 3, 5]:
        for corr in CORRUPTIONS:
            print(f"Ablation TENT steps={steps}: {model_name}, {corr['name']}")
            m = evaluate_tent(model, loader, mean, std, corr['type'], corr['intensity'], lr=1e-3, steps=steps, device=device)
            ablation_results.append({
                'Model': model_name, 'Corruption': corr['name'], 'Method': f'TENT_steps={steps}',
                'Accuracy': m['accuracy'], 'Balanced_Acc': m['balanced_accuracy'],
                'MCC': m['matthews_corrcoef'], 'F1_macro': m['f1_macro']
            })

# Ablation 3: BN‑update momentum (0.05, 0.1, 0.2, 0.5) for all corruptions
def evaluate_bn_tta_momentum(model, test_loader, norm_mean, norm_std, corruption_type, intensity, momentum, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for images, labels in tqdm(test_loader, desc=f"BN‑TTA mom={momentum}"):
        images, labels = images.to(device), labels.to(device)
        if corruption_type is not None:
            images = apply_corruption(images, corruption_type, norm_mean, norm_std, intensity)
        # Temporarily set BN momentum
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.momentum = momentum
        model.train()
        with torch.no_grad():
            _ = model(images)
        model.eval()
        with torch.no_grad():
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())
    metrics = compute_all_metrics(np.array(all_labels), np.array(all_preds), np.array(all_probs), NUM_CLASSES)
    return metrics

for model_name, model, loader, mean, std in models_to_test:
    for mom in [0.05, 0.1, 0.2, 0.5]:
        for corr in CORRUPTIONS:
            print(f"Ablation BN momentum={mom}: {model_name}, {corr['name']}")
            m = evaluate_bn_tta_momentum(model, loader, mean, std, corr['type'], corr['intensity'], mom, device)
            ablation_results.append({
                'Model': model_name, 'Corruption': corr['name'], 'Method': f'BN_mom={mom}',
                'Accuracy': m['accuracy'], 'Balanced_Acc': m['balanced_accuracy'],
                'MCC': m['matthews_corrcoef'], 'F1_macro': m['f1_macro']
            })

# Ablation 4: TTT learning rate (1e-5, 1e-4, 1e-3) and steps (1, 3, 5) for all corruptions
for model_name, model, loader, mean, std in models_to_test:
    # Use the already pre‑trained TTT models
    if model_name == 'YorubaNet':
        model_ttt_base = yorubanet_ttt
    else:
        model_ttt_base = resnet18_ttt
    for lr in [1e-5, 1e-4, 1e-3]:
        for steps in [1, 3, 5]:
            for corr in CORRUPTIONS:
                print(f"Ablation TTT lr={lr}, steps={steps}: {model_name}, {corr['name']}")
                m = evaluate_ttt(model_ttt_base, loader, mean, std, corr['type'], corr['intensity'], lr=lr, steps=steps, device=device)
                ablation_results.append({
                    'Model': model_name, 'Corruption': corr['name'], 'Method': f'TTT_lr={lr}_steps={steps}',
                    'Accuracy': m['accuracy'], 'Balanced_Acc': m['balanced_accuracy'],
                    'MCC': m['matthews_corrcoef'], 'F1_macro': m['f1_macro']
                })

# ------------------- 15. SAVE RESULTS TO CSV -------------------
df_results = pd.DataFrame(results)
df_ablation = pd.DataFrame(ablation_results)

df_results.to_csv(os.path.join(output_folder, 'comparative_results.csv'), index=False)
df_ablation.to_csv(os.path.join(output_folder, 'ablation_results.csv'), index=False)

# ------------------- 16. GENERATE PLOTS -------------------
# Plot 1: Accuracy comparison bar chart (grouped by corruption, for each model)
for model_name in ['YorubaNet', 'ResNet18']:
    df_model = df_results[df_results['Model'] == model_name]
    methods = ['No TTA', 'BN‑TTA', 'TENT', 'TTT']
    x = np.arange(len(CORRUPTIONS))
    width = 0.2
    plt.figure(figsize=(12,6))
    for i, method in enumerate(methods):
        df_m = df_model[df_model['Method'] == method]
        acc = []
        for corr in CORRUPTIONS:
            val = df_m[df_m['Corruption'] == corr['name']]['Accuracy'].values
            acc.append(val[0] if len(val) > 0 else 0)
        plt.bar(x + i*width, acc, width, label=method)
    plt.xticks(x + width*1.5, [c['name'] for c in CORRUPTIONS], rotation=45)
    plt.ylabel('Accuracy')
    plt.title(f'{model_name} - Accuracy Comparison')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{model_name}_comparison_bars.png'))
    plt.close()

# Plot 2: Improvement over baseline (for each method)
for model_name in ['YorubaNet', 'ResNet18']:
    df_model = df_results[df_results['Model'] == model_name]
    baseline = df_model[df_model['Method'] == 'No TTA'].set_index('Corruption')['Accuracy']
    improvements = {}
    for method in ['BN‑TTA', 'TENT', 'TTT']:
        df_m = df_model[df_model['Method'] == method].set_index('Corruption')
        imp = (df_m['Accuracy'] - baseline) * 100
        improvements[method] = imp
    df_imp = pd.DataFrame(improvements).reset_index()
    df_imp_melted = df_imp.melt(id_vars='Corruption', var_name='Method', value_name='Improvement (%)')
    plt.figure(figsize=(10,6))
    sns.barplot(data=df_imp_melted, x='Corruption', y='Improvement (%)', hue='Method')
    plt.axhline(0, color='red', linestyle='--')
    plt.title(f'{model_name} - Accuracy Improvement over Baseline')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{model_name}_improvement_bars.png'))
    plt.close()

# Plot 3: Ablation heatmap for TENT learning rate (averaged over corruptions)
df_ablation_tent_lr = df_ablation[df_ablation['Method'].str.startswith('TENT_lr')]
pivot = df_ablation_tent_lr.pivot_table(index=['Model', 'Corruption'], columns='Method', values='Accuracy')
plt.figure(figsize=(10,8))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='viridis')
plt.title('TENT Accuracy by Learning Rate (across corruptions)')
plt.tight_layout()
plt.savefig(os.path.join(output_folder, 'ablation_tent_lr_heatmap.png'))
plt.close()

# Plot 4: Ablation line plot for BN momentum
df_ablation_bn_mom = df_ablation[df_ablation['Method'].str.startswith('BN_mom')]
for model_name in ['YorubaNet', 'ResNet18']:
    plt.figure(figsize=(10,6))
    df_m = df_ablation_bn_mom[df_ablation_bn_mom['Model'] == model_name]
    for corr in CORRUPTIONS:
        df_c = df_m[df_m['Corruption'] == corr['name']]
        df_c = df_c.sort_values('Method')
        moms = [float(m.split('=')[1]) for m in df_c['Method']]
        acc = df_c['Accuracy'].values
        plt.plot(moms, acc, marker='o', label=corr['name'])
    plt.xlabel('BN momentum')
    plt.ylabel('Accuracy')
    plt.title(f'{model_name} - BN momentum ablation')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_folder, f'{model_name}_bn_momentum_ablation.png'))
    plt.close()

# ------------------- 17. FINAL SUMMARY -------------------
print("\n" + "="*60)
print(f"COMPARATIVE STUDY AND ABLATION COMPLETED.")
print(f"All results saved to: {output_folder}")
print("Files generated:")
print(" - comparative_results.csv : accuracy and metrics for all methods")
print(" - ablation_results.csv : hyperparameter ablation for all corruptions")
print(" - *_comparison_bars.png : bar charts of accuracy")
print(" - *_improvement_bars.png : improvement over baseline")
print(" - ablation_tent_lr_heatmap.png : heatmap of TENT LR effect")
print(" - *_bn_momentum_ablation.png : line plots for BN momentum")
print("="*60)

# Print quick summary table
print("\n=== ACCURACY COMPARISON (all methods, all corruptions) ===")
print(df_results[['Model', 'Corruption', 'Method', 'Accuracy']].to_string(index=False))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ablation results will be saved to: /content/drive/MyDrive/TTA_Comparative_Ablation/ablation_20260404_231337
Using device: cuda
Loading pre‑trained models...
YorubaNet loaded.
ResNet-18 loaded.

Pre‑training rotation heads for TTT...


Pre‑train rotation head epoch 1: 100%|██████████| 69/69 [00:04<00:00, 14.80it/s]


Epoch 1: Loss=1.0828, Acc=61.00%


Pre‑train rotation head epoch 2: 100%|██████████| 69/69 [00:04<00:00, 14.72it/s]


Epoch 2: Loss=0.6388, Acc=75.01%


...

Ablation TTT lr=0.001, steps=5: ResNet18, Salt-Pepper Noise (5%)


TTT noise: 100%|██████████| 16/16 [00:04<00:00,  3.55it/s]


Ablation TTT lr=0.001, steps=5: ResNet18, Thinning (threshold=0.3)


TTT thinning: 100%|██████████| 16/16 [00:04<00:00,  3.56it/s]



COMPARATIVE STUDY AND ABLATION COMPLETED.
All results saved to: /content/drive/MyDrive/TTA_Comparative_Ablation/ablation_20260404_231337
Files generated:
 - comparative_results.csv : accuracy and metrics for all methods
 - ablation_results.csv : hyperparameter ablation for all corruptions
 - *_comparison_bars.png : bar charts of accuracy
 - *_improvement_bars.png : improvement over baseline
 - ablation_tent_lr_heatmap.png : heatmap of TENT LR effect
 - *_bn_momentum_ablation.png : line plots for BN momentum

=== ACCURACY COMPARISON (all methods, all corruptions) ===
    Model               Corruption Method  Accuracy
YorubaNet      Gaussian Blur (σ=2) No TTA  0.793247
YorubaNet      Gaussian Blur (σ=2) BN‑TTA  0.763636
YorubaNet      Gaussian Blur (σ=2)   TENT  0.450390
YorubaNet      Gaussian Blur (σ=2)    TTT  0.447792
YorubaNet   Salt-Pepper Noise (5%) No TTA  0.072208
YorubaNet   Salt-Pepper Noise (5%) BN‑TTA  0.218182
YorubaNet   Salt-Pepper Noise (5%)   TENT  0.324156
YorubaNet 